# Capítulo 6. k vecinos más cercanos (k-NN)

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# k vecinos más cercanos, k-NN

La formulación matemática de **distancias, escalamiento y vecinos más cercanos** se desarrolla con mayor profundidad
en los capítulos 2 y 12 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de explicar k-NN, preparar datos para este método, estandarizar variables y evaluar el desempeño.

## Idea intuitiva

k-NN clasifica una nueva observación según la clase más común entre sus vecinos más cercanos.


In [ ]:
library(ggplot2)
library(class)
library(dplyr)
library(tidyr)
library(readr)
source("util_graficas.R")

datos_ejemplo <- data.frame(
  hora = c(8, 9, 10, 18, 19, 20, 22, 23),
  mes = c(1, 1, 2, 6, 6, 7, 12, 12),
  clase = c("Solo daños", "Solo daños", "Solo daños", "Con víctimas", "Con víctimas", "Con víctimas", "Con víctimas", "Con víctimas")
)

ggplot(datos_ejemplo, aes(x = hora, y = mes, color = clase)) +
  geom_point(size = 4) +
  escala_clases_color() +
  labs(title = "Ejemplo intuitivo de k-NN", x = "Hora", y = "Mes", color = "Clase") +
  tema_libro()


## Explicación del código
Se construyen puntos artificiales para visualizar la idea de vecinos cercanos.

## Distancia euclidiana

$$
d(A,B) = \sqrt{(x_1-x_2)^2 + (y_1-y_2)^2}
$$


In [ ]:
sqrt((8 - 10)^2 + (1 - 2)^2)


## Interpretación del resultado
Una distancia menor indica mayor similitud entre dos observaciones.

## Cargar y preparar datos


In [ ]:
ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  set.seed(123)
  atus_knn_base <- atus_ml |>
    sample_n(min(12000, nrow(atus_ml))) |>
    mutate(
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños")),
      MES = as.factor(MES),
      ID_HORA = as.numeric(ID_HORA),
      DIASEMANA = as.factor(DIASEMANA),
      TIPACCID = as.factor(TIPACCID),
      CAUSAACCI = as.factor(CAUSAACCI)
    ) |>
    na.omit()
}


## Explicación del código
Se toma una muestra para que k-NN sea rápido y reproducible. Este método puede ser costoso en bases grandes.

## Modelo k-NN


In [ ]:
if (exists("atus_knn_base")) {
  matriz_predictoras <- model.matrix(~ ID_HORA + MES + DIASEMANA + TIPACCID + CAUSAACCI - 1, data = atus_knn_base)
  y <- atus_knn_base$accidente_con_victimas

  set.seed(123)
  idx <- sample(1:nrow(matriz_predictoras), size = round(0.7 * nrow(matriz_predictoras)))
  x_entrenamiento <- matriz_predictoras[idx, ]
  x_prueba <- matriz_predictoras[-idx, ]
  y_entrenamiento <- y[idx]
  y_prueba <- y[-idx]

  medias <- apply(x_entrenamiento, 2, mean)
  desviaciones <- apply(x_entrenamiento, 2, sd)
  desviaciones[desviaciones == 0] <- 1

  x_entrenamiento_esc <- scale(x_entrenamiento, center = medias, scale = desviaciones)
  x_prueba_esc <- scale(x_prueba, center = medias, scale = desviaciones)

  pred_knn_5 <- knn(x_entrenamiento_esc, x_prueba_esc, y_entrenamiento, k = 5)
  table(Real = y_prueba, Predicho = pred_knn_5)
}


## Explicación del código
Se crean variables dummy, se estandarizan las columnas y se clasifica con k igual a 5.

## Comparación de valores de k


In [ ]:
calcular_metricas <- function(real, predicho) {
  real <- factor(real, levels = c("Con víctimas", "Solo daños"))
  predicho <- factor(predicho, levels = c("Con víctimas", "Solo daños"))
  m <- table(Real = real, Predicho = predicho)
  VP <- m["Con víctimas", "Con víctimas"]
  FN <- m["Con víctimas", "Solo daños"]
  FP <- m["Solo daños", "Con víctimas"]
  VN <- m["Solo daños", "Solo daños"]
  data.frame(exactitud=(VP+VN)/(VP+FN+FP+VN), sensibilidad=VP/(VP+FN), especificidad=VN/(VN+FP))
}

if (exists("x_entrenamiento_esc")) {
  valores_k <- c(3, 5, 11)
  resultados_knn <- data.frame()

  for (k_actual in valores_k) {
    pred_actual <- knn(x_entrenamiento_esc, x_prueba_esc, y_entrenamiento, k = k_actual)
    tmp <- calcular_metricas(y_prueba, pred_actual)
    tmp$k <- k_actual
    resultados_knn <- rbind(resultados_knn, tmp)
  }

  resultados_knn <- resultados_knn |> select(k, exactitud, sensibilidad, especificidad)
  resultados_knn
}

if (exists("resultados_knn")) {
  resultados_largos <- resultados_knn |>
    pivot_longer(cols = c(exactitud, sensibilidad, especificidad), names_to = "metrica", values_to = "valor")

  ggplot(resultados_largos, aes(x = k, y = valor, color = metrica, group = metrica)) +
    geom_line(linewidth = 1.1) +
    geom_point(size = 3) +
    scale_color_manual(values = c(exactitud = col_azul, sensibilidad = col_coral, especificidad = col_turquesa)) +
    labs(title = "Comparación de métricas para distintos valores de k", x = "Valor de k", y = "Valor") +
    tema_libro()
}


## Interpretación del resultado
Cambiar k modifica el balance entre exactitud, sensibilidad y especificidad. No existe un k universalmente mejor.

## Laboratorio interactivo: k-NN

Selecciona el número de vecinos y mueve un punto nuevo. El laboratorio identifica sus vecinos más cercanos y muestra la clase predicha.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El lector puede cambiar el valor de $k$, mover una observación nueva y visualizar sus vecinos, distancias y clase predicha.

## Caso aplicado B: k-NN con COVID-19

En esta segunda ruta aplicada usamos la misma muestra educativa de COVID-19 México 2022 empleada en los capítulos anteriores. El objetivo es clasificar `MURIO` a partir de edad, neumonía, diabetes, hipertensión, obesidad, enfermedad renal crónica y número de comorbilidades.

> **Uso académico:** este ejercicio permite estudiar el comportamiento de k-NN. No es una calculadora clínica ni un instrumento de diagnóstico.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_knn <- read_csv(ruta_covid, show_col_types = FALSE) |>
    select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
           OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    drop_na()

  set.seed(2026)
  covid_knn <- covid_knn |>
    sample_n(min(12000, nrow(covid_knn))) |>
    mutate(
      MURIO = factor(MURIO, levels = c(1, 0), labels = c("Defunción", "Sin defunción"))
    )

  X_covid <- covid_knn |> select(-MURIO) |> as.matrix()
  y_covid <- covid_knn$MURIO

  set.seed(2026)
  idx_covid <- sample(seq_len(nrow(X_covid)), size = floor(0.80 * nrow(X_covid)))

  x_train_covid <- X_covid[idx_covid, , drop = FALSE]
  x_test_covid <- X_covid[-idx_covid, , drop = FALSE]
  y_train_covid <- y_covid[idx_covid]
  y_test_covid <- y_covid[-idx_covid]

  medias_covid <- apply(x_train_covid, 2, mean)
  desv_covid <- apply(x_train_covid, 2, sd)
  desv_covid[desv_covid == 0] <- 1

  x_train_covid_z <- scale(x_train_covid, center = medias_covid, scale = desv_covid)
  x_test_covid_z <- scale(x_test_covid, center = medias_covid, scale = desv_covid)
}


La estandarización es especialmente importante en k-NN porque la distancia euclidiana sería dominada por variables con escalas mayores, como la edad.


In [ ]:
if (exists("x_train_covid_z")) {
  pred_covid_knn <- knn(train = x_train_covid_z, test = x_test_covid_z, cl = y_train_covid, k = 11)

  matriz_covid_knn <- table(
    Real = factor(y_test_covid, levels = c("Defunción", "Sin defunción")),
    Predicho = factor(pred_covid_knn, levels = c("Defunción", "Sin defunción"))
  )
  matriz_covid_knn
}


In [ ]:
if (exists("x_train_covid_z")) {
  evaluar_knn_covid <- function(k) {
    p <- knn(x_train_covid_z, x_test_covid_z, y_train_covid, k = k)
    m <- table(
      Real = factor(y_test_covid, levels = c("Defunción", "Sin defunción")),
      Predicho = factor(p, levels = c("Defunción", "Sin defunción"))
    )
    VP <- m[1,1]; FN <- m[1,2]; FP <- m[2,1]; VN <- m[2,2]
    data.frame(
      k = k,
      exactitud = (VP + VN) / sum(m),
      sensibilidad = ifelse(VP + FN == 0, NA, VP / (VP + FN)),
      especificidad = ifelse(VN + FP == 0, NA, VN / (VN + FP))
    )
  }

  resultados_k_covid <- do.call(rbind, lapply(c(3, 5, 11, 21), evaluar_knn_covid))
  resultados_k_covid
}


## Interpretación
El valor de $k$ controla cuánto se suaviza la decisión. Valores pequeños reaccionan más a observaciones locales; valores mayores producen fronteras más estables. En datos desbalanceados conviene mirar sensibilidad y especificidad, no solo exactitud.

## Materiales complementarios del capítulo

### Video del capítulo

### Video del capítulo

Disponible en YouTube:

<https://youtu.be/NLJj0QAWomU>

| Recurso | Descripción | Abrir o descargar |
|---|---|---|
| Presentación en PDF | Síntesis del capítulo para lectura o exposición. | [Abrir PDF](recursos/capitulo-06/capitulo-06-knn-presentacion.pdf) |
| Presentación editable | Diapositivas en PowerPoint. | [Descargar PPTX](recursos/capitulo-06/capitulo-06-knn-presentacion.pptx) |
| Infografía | Resumen visual del algoritmo k-NN. | [Abrir infografía](recursos/capitulo-06/capitulo-06-knn-infografia.png) |

![Infografía del capítulo 6](recursos/capitulo-06/capitulo-06-knn-infografia.png)

Los materiales fueron creados con apoyo de NotebookLM de Google a partir del
contenido del libro y revisados y adaptados por el autor.

## Conclusión

k-NN es un método intuitivo basado en similitud. Su desempeño depende del valor de $k$, del escalamiento y de la calidad de las variables predictoras.

## Referencias fundamentales de k-NN

El método de los vecinos más cercanos tiene como antecedente clásico el informe de @fix1951discriminatory. Sus propiedades de clasificación y cotas de error fueron estudiadas por @cover1967nearest. Una exposición moderna, acompañada de aplicaciones en R, puede consultarse en @james2021islr.
